# Basic Transformer Univariate Recursive Multi-Step Forecasting

Este notebook implementa la misma logica del notebook recursivo anterior, pero usando un
Transformer basico como modelo univariado por variable.

Incluye:

- entrenamiento one-step por variable
- inferencia multi-step recursiva
- 10 runs
- metricas RMSE, MAE, MAPE y training time
- exporte a Excel por run, iteracion y variable
- hojas extra con metricas por horizonte

Estructura del Excel generado:

- `Transformer_summary`
- `Transformer_horizon`
- `configuration`


## 1. Dependencias

Si el entorno no tiene las librerias necesarias, instala primero las dependencias.


In [ ]:
# Ejecuta esta celda solo si tu entorno no tiene las dependencias instaladas.
# %pip install pandas matplotlib tensorflow openpyxl scikit-learn

## 2. Imports y configuracion

El Transformer se mantiene univariado para que los resultados sean comparables con
BiGRU, LSTM y ARIMA.


In [ ]:
from pathlib import Path
import random
import warnings
from time import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling1D,
    Input,
    Layer,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.metrics import MeanAbsoluteError, MeanAbsolutePercentageError
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-white")
pd.set_option("display.max_columns", 100)

DATA_FILE = Path("data.csv")
OUTPUT_DIR = Path("outputs_recursive_multistep_transformer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_PATH = OUTPUT_DIR / "metrics_summary.xlsx"

RUNS = 10
NUM_FOLDS = 4
TRAIN_WITHIN_FOLD_RATIO = 0.8
WINDOW_SIZE = 10
FORECAST_HORIZON = 12
BASE_SEED = 42

TARGET_COLUMNS = None  # None -> usa todas las columnas
CUTOFF_DATE = "2023-07-05 00:00:00"

TRANSFORMER_CONFIG = {
    "d_model": 64,
    "num_blocks": 2,
    "head_size": 16,
    "num_heads": 4,
    "ff_dim": 64,
    "dropout": 0.1,
    "dense_units": 32,
    "learning_rate": 1e-4,
    "epochs": 100,
    "batch_size": 128,
    "patience": 5,
}

print(f"Archivo de datos: {DATA_FILE.resolve()}")
print(f"Directorio de salida: {OUTPUT_DIR.resolve()}")
print(f"Excel de metricas: {EXCEL_PATH.resolve()}")

## 3. Funciones auxiliares

Se definen funciones para:

- reproducibilidad
- folds temporales
- escalado univariado
- ventanas para Transformer
- modelo Transformer basico
- forecast recursivo
- metricas y exporte a Excel


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


class PositionalEncoding(Layer):
    def __init__(self, max_len=5000, d_model=64, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model

        positions = np.arange(max_len)[:, np.newaxis]
        dims = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
        angle_rads = positions * angle_rates

        positional_matrix = np.zeros((max_len, d_model))
        positional_matrix[:, 0::2] = np.sin(angle_rads[:, 0::2])
        positional_matrix[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.positional_encoding = tf.constant(positional_matrix[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        return inputs + self.positional_encoding[:, :seq_len, :]


def root_mean_squared_error(y_true, y_pred):
    return tf.math.sqrt(tf.math.reduce_mean(tf.math.square(y_pred - y_true)))


def transformer_encoder(x, head_size=16, num_heads=4, ff_dim=64, dropout=0.1):
    attn = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout,
    )(x, x)
    x = LayerNormalization(epsilon=1e-6)(x + Dropout(dropout)(attn))

    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dropout(dropout)(ff)
    ff = Dense(int(x.shape[-1]))(ff)

    return LayerNormalization(epsilon=1e-6)(x + ff)


def build_transformer_model(window_size: int, config: dict):
    inputs = Input(shape=(window_size, 1))

    x = Dense(config["d_model"])(inputs)
    x = PositionalEncoding(max_len=window_size, d_model=config["d_model"])(x)

    for _ in range(config["num_blocks"]):
        x = transformer_encoder(
            x,
            head_size=config["head_size"],
            num_heads=config["num_heads"],
            ff_dim=config["ff_dim"],
            dropout=config["dropout"],
        )

    x = GlobalAveragePooling1D()(x)
    x = Dense(config["dense_units"], activation="relu")(x)
    x = Dropout(config["dropout"])(x)
    outputs = Dense(1, activation="linear")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(learning_rate=config["learning_rate"]),
        loss=root_mean_squared_error,
        metrics=[MeanAbsoluteError(), MeanAbsolutePercentageError()],
    )
    return model


def build_temporal_folds(df: pd.DataFrame, num_folds: int, train_ratio: float):
    folds = {}
    cut_dates = {}
    len_data = len(df) / num_folds

    for iteration, _ in enumerate(range(0, num_folds - 1)):
        len_train_data = int(len_data * (iteration + 1))
        train_size = int(len_train_data * train_ratio)
        val_size = int(len_train_data * (1 - train_ratio))

        tr = df.iloc[:train_size].copy()
        vl = df.iloc[train_size: train_size + val_size].copy()
        ts = df.iloc[len_train_data: len_train_data + int(len_data)].copy()

        folds[iteration] = {
            "train": tr,
            "val": vl,
            "test": ts,
        }
        cut_dates[iteration] = {
            "train_end": tr.index[-1],
            "val_end": vl.index[-1],
            "test_end": ts.index[-1],
        }

    return folds, cut_dates


def create_transformer_windows(series_scaled, window_size: int):
    x_data = []
    y_data = []
    for idx in range(len(series_scaled) - window_size):
        x_data.append(series_scaled[idx: idx + window_size])
        y_data.append(series_scaled[idx + window_size])

    x_array = np.array(x_data, dtype=np.float32).reshape(-1, window_size, 1)
    y_array = np.array(y_data, dtype=np.float32).reshape(-1, 1)
    return x_array, y_array


def prepare_univariate_fold_data(tr_series, vl_series, ts_series, window_size: int, forecast_horizon: int):
    scaler = MinMaxScaler(feature_range=(-1, 1))

    train_raw = tr_series.to_numpy(dtype=float).reshape(-1, 1)
    val_raw = vl_series.to_numpy(dtype=float).reshape(-1, 1)
    test_raw = ts_series.to_numpy(dtype=float).reshape(-1, 1)

    scaler.fit(train_raw)

    train_scaled = scaler.transform(train_raw).reshape(-1)
    val_scaled = scaler.transform(val_raw).reshape(-1)
    test_scaled = scaler.transform(test_raw).reshape(-1)

    x_tr, y_tr = create_transformer_windows(train_scaled, window_size)
    x_vl, y_vl = create_transformer_windows(val_scaled, window_size)

    history_before_test_scaled = np.concatenate([train_scaled, val_scaled])
    available_horizon = min(forecast_horizon, len(test_raw))

    return {
        "scaler": scaler,
        "train_raw": train_raw.reshape(-1),
        "val_raw": val_raw.reshape(-1),
        "test_raw": test_raw.reshape(-1),
        "train_scaled": train_scaled,
        "val_scaled": val_scaled,
        "test_scaled": test_scaled,
        "x_tr": x_tr,
        "y_tr": y_tr,
        "x_vl": x_vl,
        "y_vl": y_vl,
        "history_before_test_scaled": history_before_test_scaled,
        "actual_future": test_raw.reshape(-1)[:available_horizon],
        "available_horizon": available_horizon,
    }


def inverse_transform_vector(scaler: MinMaxScaler, values_1d):
    values_2d = np.array(values_1d, dtype=float).reshape(-1, 1)
    return scaler.inverse_transform(values_2d).reshape(-1)


def recursive_forecast_transformer(model, scaler, history_scaled, horizon: int, window_size: int):
    if len(history_scaled) < window_size:
        raise ValueError("No hay suficientes observaciones historicas para iniciar el forecast recursivo.")

    history = list(np.array(history_scaled, dtype=float).reshape(-1))
    predictions_scaled = []

    for _ in range(horizon):
        window = np.array(history[-window_size:], dtype=np.float32).reshape(1, window_size, 1)
        next_scaled = float(model.predict(window, verbose=0).reshape(-1)[0])
        predictions_scaled.append(next_scaled)
        history.append(next_scaled)

    return inverse_transform_vector(scaler, predictions_scaled)


def compute_summary_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float).reshape(-1)
    y_pred = np.array(y_pred, dtype=float).reshape(-1)

    rmse = float(np.sqrt(np.mean(np.square(y_true - y_pred))))
    mae = float(np.mean(np.abs(y_true - y_pred)))

    denominator = np.where(np.abs(y_true) < 1e-8, np.nan, np.abs(y_true))
    mape = float(np.nanmean(np.abs((y_true - y_pred) / denominator)) * 100.0)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
    }


def compute_horizon_rows(model_name, run_id, iteration_id, variable_name, y_true, y_pred):
    rows = []
    y_true = np.array(y_true, dtype=float).reshape(-1)
    y_pred = np.array(y_pred, dtype=float).reshape(-1)

    for horizon_idx, (true_value, pred_value) in enumerate(zip(y_true, y_pred), start=1):
        point_metrics = compute_summary_metrics([true_value], [pred_value])
        rows.append({
            "model": model_name,
            "run": run_id,
            "iteration": iteration_id,
            "variable": variable_name,
            "horizon": horizon_idx,
            "RMSE": point_metrics["RMSE"],
            "MAE": point_metrics["MAE"],
            "MAPE": point_metrics["MAPE"],
        })
    return rows


def make_prepared_folds(folds, target_columns, window_size: int, forecast_horizon: int):
    prepared = {}
    for iteration_id, fold_data in folds.items():
        prepared[iteration_id] = {}
        for col in target_columns:
            prepared[iteration_id][col] = prepare_univariate_fold_data(
                fold_data["train"][col],
                fold_data["val"][col],
                fold_data["test"][col],
                window_size=window_size,
                forecast_horizon=forecast_horizon,
            )
    return prepared


def export_metrics_to_excel(summary_rows, horizon_rows, excel_path: Path, config_dict: dict):
    summary_df = pd.DataFrame(summary_rows)
    horizon_df = pd.DataFrame(horizon_rows)

    if not summary_df.empty:
        summary_df = summary_df.sort_values(["run", "iteration", "variable"]).reset_index(drop=True)
    if not horizon_df.empty:
        horizon_df = horizon_df.sort_values(["run", "iteration", "variable", "horizon"]).reset_index(drop=True)

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Transformer_summary", index=False)
        horizon_df.to_excel(writer, sheet_name="Transformer_horizon", index=False)
        pd.DataFrame([config_dict]).to_excel(writer, sheet_name="configuration", index=False)


def plot_example_forecast(actual_values, predicted_values, title: str):
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(actual_values) + 1), actual_values, label="Real", linewidth=2.5)
    plt.plot(range(1, len(predicted_values) + 1), predicted_values, label="Forecast", linewidth=2.5)
    plt.xlabel("Horizon")
    plt.ylabel("Value")
    plt.title(title)
    plt.grid(False)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 4. Carga de datos

Se usa el mismo `data.csv` del proyecto y el mismo recorte temporal que el resto de notebooks comparables.


In [ ]:
new_order_columns = [
    "Free Memory",
    "Used Memory",
    "Free Disk",
    "Used Disk",
    "Disk read/s",
    "Disk write/s",
    "NetBytes In",
    "NetBytes Out",
    "NetPackets In",
    "NetPackets Out",
    "Rx packets",
    "Tx packets",
    "CPU percent",
    "Memory Used percent",
    "Disk Used percent",
    "Uptime",
]

df = pd.read_csv(DATA_FILE, index_col=0, parse_dates=["beginTimeSeconds"])
df = df[new_order_columns].copy()
df = df[df.index <= CUTOFF_DATE].copy()

if TARGET_COLUMNS is None:
    target_columns = list(df.columns)
else:
    target_columns = [col for col in TARGET_COLUMNS if col in df.columns]

print(df.shape)
print(df.index.min(), df.index.max())
print(target_columns)
df.head()

## 5. Folds temporales

Se mantiene el esquema expandible con train y validation dentro del bloque historico y el siguiente bloque como test.


In [ ]:
folds, cut_dates = build_temporal_folds(
    df=df,
    num_folds=NUM_FOLDS,
    train_ratio=TRAIN_WITHIN_FOLD_RATIO,
)

fold_rows = []
for iteration_id, fold_data in folds.items():
    fold_rows.append({
        "iteration": iteration_id,
        "train_start": fold_data["train"].index.min(),
        "train_end": fold_data["train"].index.max(),
        "val_start": fold_data["val"].index.min(),
        "val_end": fold_data["val"].index.max(),
        "test_start": fold_data["test"].index.min(),
        "test_end": fold_data["test"].index.max(),
        "train_rows": len(fold_data["train"]),
        "val_rows": len(fold_data["val"]),
        "test_rows": len(fold_data["test"]),
    })

pd.DataFrame(fold_rows)

## 6. Preparacion univariada por fold

Para cada variable se ajusta un scaler propio, se construyen ventanas para el Transformer y se deja lista la historia para el forecast recursivo.


In [ ]:
prepared_folds = make_prepared_folds(
    folds=folds,
    target_columns=target_columns,
    window_size=WINDOW_SIZE,
    forecast_horizon=FORECAST_HORIZON,
)

preview_rows = []
for iteration_id, variables_data in prepared_folds.items():
    sample_col = target_columns[0]
    sample = variables_data[sample_col]
    preview_rows.append({
        "iteration": iteration_id,
        "variable": sample_col,
        "x_tr_shape": sample["x_tr"].shape,
        "y_tr_shape": sample["y_tr"].shape,
        "x_vl_shape": sample["x_vl"].shape,
        "y_vl_shape": sample["y_vl"].shape,
        "available_horizon": sample["available_horizon"],
    })

pd.DataFrame(preview_rows)

## 7. Experimentos Transformer

El modelo se entrena one-step y la generacion multi-step se hace de forma recursiva.

Se guardan:

- metricas agregadas por `run x iteration x variable`
- metricas por horizonte
- `training_time_seconds`
- `epochs_trained`
- `best_val_loss`


In [ ]:
def run_transformer_experiments(prepared_folds, target_columns, runs, base_seed, cut_dates, transformer_config):
    summary_records = []
    horizon_records = []
    example_forecasts = {}

    for run_id in range(1, runs + 1):
        print(f"===== Transformer | run {run_id}/{runs} =====")

        for iteration_id, variables_data in prepared_folds.items():
            print(f"--- Iteration {iteration_id} ---")

            for variable_idx, variable_name in enumerate(target_columns):
                prepared = variables_data[variable_name]
                current_seed = base_seed + run_id * 1000 + iteration_id * 100 + variable_idx * 10 + 7
                set_seed(current_seed)
                tf.keras.backend.clear_session()

                x_tr = prepared["x_tr"]
                y_tr = prepared["y_tr"]
                x_vl = prepared["x_vl"]
                y_vl = prepared["y_vl"]

                if len(x_tr) == 0:
                    continue

                model = build_transformer_model(WINDOW_SIZE, transformer_config)
                callbacks = []
                validation_data = None
                if len(x_vl) > 0:
                    validation_data = (x_vl, y_vl)
                    callbacks.append(
                        tf.keras.callbacks.EarlyStopping(
                            monitor="val_loss",
                            patience=transformer_config["patience"],
                            restore_best_weights=True,
                        )
                    )

                start_time = time()
                history = model.fit(
                    x=x_tr,
                    y=y_tr,
                    validation_data=validation_data,
                    epochs=transformer_config["epochs"],
                    batch_size=transformer_config["batch_size"],
                    verbose=0,
                    callbacks=callbacks,
                )
                training_time = time() - start_time

                available_horizon = prepared["available_horizon"]
                predicted = recursive_forecast_transformer(
                    model=model,
                    scaler=prepared["scaler"],
                    history_scaled=prepared["history_before_test_scaled"],
                    horizon=available_horizon,
                    window_size=WINDOW_SIZE,
                )
                actual = prepared["actual_future"][:available_horizon]

                val_losses = history.history.get("val_loss", [])
                best_val_loss = float(min(val_losses)) if val_losses else np.nan

                metrics = compute_summary_metrics(actual, predicted)
                summary_records.append({
                    "model": "Transformer",
                    "run": run_id,
                    "iteration": iteration_id,
                    "variable": variable_name,
                    "forecast_horizon": available_horizon,
                    "train_end": cut_dates[iteration_id]["train_end"],
                    "val_end": cut_dates[iteration_id]["val_end"],
                    "test_end": cut_dates[iteration_id]["test_end"],
                    "RMSE": metrics["RMSE"],
                    "MAE": metrics["MAE"],
                    "MAPE": metrics["MAPE"],
                    "training_time_seconds": training_time,
                    "epochs_trained": len(history.history.get("loss", [])),
                    "best_val_loss": best_val_loss,
                    "seed": current_seed,
                })

                horizon_records.extend(
                    compute_horizon_rows(
                        model_name="Transformer",
                        run_id=run_id,
                        iteration_id=iteration_id,
                        variable_name=variable_name,
                        y_true=actual,
                        y_pred=predicted,
                    )
                )

                example_key = (iteration_id, variable_name)
                if example_key not in example_forecasts:
                    example_forecasts[example_key] = {
                        "actual": actual,
                        "predicted": predicted,
                    }

    return {
        "summary": summary_records,
        "horizon": horizon_records,
        "examples": example_forecasts,
    }


transformer_results = run_transformer_experiments(
    prepared_folds=prepared_folds,
    target_columns=target_columns,
    runs=RUNS,
    base_seed=BASE_SEED,
    cut_dates=cut_dates,
    transformer_config=TRANSFORMER_CONFIG,
)

print("Transformer summary rows:", len(transformer_results["summary"]))

## 8. Exporte a Excel

Se genera un unico archivo con hojas de resumen y de detalle por horizonte.


In [ ]:
config_export = {
    "data_file": str(DATA_FILE),
    "runs": RUNS,
    "num_folds": NUM_FOLDS,
    "train_within_fold_ratio": TRAIN_WITHIN_FOLD_RATIO,
    "window_size": WINDOW_SIZE,
    "forecast_horizon": FORECAST_HORIZON,
    "cutoff_date": CUTOFF_DATE,
    "target_columns": ", ".join(target_columns),
    "transformer_config": str(TRANSFORMER_CONFIG),
}

export_metrics_to_excel(
    summary_rows=transformer_results["summary"],
    horizon_rows=transformer_results["horizon"],
    excel_path=EXCEL_PATH,
    config_dict=config_export,
)

print(f"Excel generado en: {EXCEL_PATH.resolve()}")

## 9. Vista rapida de resultados


In [ ]:
transformer_summary_df = pd.DataFrame(transformer_results["summary"])
display(transformer_summary_df.head())

In [ ]:
example_variable = target_columns[0]
example_iteration = 0

plot_example_forecast(
    actual_values=transformer_results["examples"][(example_iteration, example_variable)]["actual"],
    predicted_values=transformer_results["examples"][(example_iteration, example_variable)]["predicted"],
    title=f"Transformer recursive multi-step | iteration {example_iteration} | {example_variable}",
)